# Chronos-Bolt zero-shot residual probe

**Idea.** A structural break means `x_online` stops behaving like `x_hist`. So forecast the start of the online segment *from* the historical segment with a pretrained forecaster (no training), then measure how badly reality violates that forecast. Large standardized residuals ⇒ the online stream left the historical regime ⇒ break.

One forward pass per series (cheap, budget-safe), no labels used to fit anything. Self-contained: raw arrays from `tools.py`, metric inlined. Nothing outside this folder imported.

**Signal per series**
```
context  = last CTX points of x_hist
forecast = Chronos-Bolt quantiles for next H = min(len(online), HORIZON)
r_t      = |x_online[t] - q50_t| / ((q90_t - q10_t)/2 + eps)   # ~std residual
score    = mean_t r_t                                          # break-ness
```

**Eval.** series-level ROC-AUC (what the signal ranks) + TS-AUC after broadcasting the per-series score across its online steps, comparable to the 0.6064 per-step bank baseline.

## Config

In [1]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # torch+lgb OpenMP clash guard

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score

from tools import emit_train_datasets, PATH

MODEL    = "amazon/chronos-bolt-small"   # small=48M; -mini/-base also available
CTX      = 512                           # historical context fed to the model
HORIZON  = 64                            # Chronos-Bolt max prediction length
QLEVELS  = [0.1, 0.5, 0.9]
N_SERIES = 500                           # subset for a quick probe; None = all 10k
BATCH    = 32
EPS      = 1e-6
DEVICE   = ("mps" if torch.backends.mps.is_available()
            else "cuda" if torch.cuda.is_available() else "cpu")
print("device", DEVICE)

device mps


## Inlined TS-AUC (per online-step stratified AUC)

In [11]:
from tools import local_ts_auc

## Load data (subset)

In [15]:
y_train_index = pd.read_parquet(f"{PATH}y_train_index.parquet")   # id -> tau_index, tau
ids = list(y_train_index.index)[:N_SERIES]
id_set = set(ids)

X = pd.read_parquet(f"{PATH}X_train.parquet")
X = X[X.index.get_level_values("id").isin(id_set)]
print("series", len(ids), "| break_rate", (y_train_index.loc[ids, "tau_index"] != -1).mean())

series 500 | break_rate 0.52


## Load Chronos-Bolt (once)

In [2]:
from chronos import BaseChronosPipeline

pipeline = BaseChronosPipeline.from_pretrained(
    MODEL, device_map=DEVICE, torch_dtype=torch.float32,
)
print("loaded", MODEL)

/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 143/143 [00:00<00:00, 28572.10it/s]

loaded amazon/chronos-bolt-small


## Residual featurizer

In [16]:
def series_residual(x_hist, x_online):
    """Rolling chunked forecast over the WHOLE online segment.

    For each 64-step chunk we forecast from all history up to the chunk start
    (hist + earlier online, teacher-forced true values -> causal). The break
    shows up as a residual spike wherever tau is, not just in the first 64 steps.

    Returns p: per-online-step cumulative break evidence (expanding max of the
    standardized residual). Monotone non-decreasing, so it matches the target
    that turns on at tau and stays on.
    """
    n = len(x_online)
    if n == 0:
        return np.zeros(0)
    hist = np.asarray(x_hist, np.float64)
    onl  = np.asarray(x_online, np.float64)
    full = np.concatenate([hist, onl])
    starts = list(range(0, n, HORIZON))
    # one context per chunk = history up to that chunk's start, last CTX points
    ctxs = [torch.tensor(full[:len(hist) + s][-CTX:], dtype=torch.float32) for s in starts]
    q, _ = pipeline.predict_quantiles(
        inputs=ctxs, prediction_length=HORIZON, quantile_levels=QLEVELS,
    )  # [n_chunks, HORIZON, 3]
    q = q.cpu().numpy()
    q10, q50, q90 = q[..., 0], q[..., 1], q[..., 2]
    scale = (q90 - q10) / 2.0 + EPS
    r = np.zeros(n)
    for i, s in enumerate(starts):
        h = min(HORIZON, n - s)
        r[s:s + h] = np.abs(onl[s:s + h] - q50[i, :h]) / scale[i, :h]
    return np.maximum.accumulate(r)   # cumulative evidence

## Build features (one forward pass per batch of series)

In [17]:
series_score, series_label = {}, {}
rows_id, rows_time, rows_pred = [], [], []   # genuine per-step predictions
ids = list(y_train_index.index)[:N_SERIES]

n_done = 0
for sid, x_hist, x_online, tau in emit_train_datasets(X, y_train_index, selected_ids=ids):
    p = series_residual(x_hist, x_online)          # per-online-step evidence
    series_score[sid] = float(p[-1]) if len(p) else 0.0   # end-of-stream score
    series_label[sid] = int(tau is not None)
    sub = X.loc[sid]
    on_times = sub.index[sub["period"].to_numpy() == 2]
    rows_id.extend([sid] * len(on_times))
    rows_time.extend(list(on_times))
    rows_pred.extend(p.tolist())
    n_done += 1
    if n_done % 50 == 0:
        print(f"  {n_done}/{len(ids)}")
print("done", n_done)

  50/500
  100/500
  150/500
  200/500
  250/500
  300/500
  350/500
  400/500
  450/500
  500/500
done 500


## Eval — series-level ROC-AUC (what the residual signal ranks)

In [18]:
s = np.array([series_score[i] for i in ids])
y = np.array([series_label[i] for i in ids])
print(f"series-level ROC-AUC: {roc_auc_score(y, s):.4f}  "
      f"(n={len(ids)}, break_rate={y.mean():.3f})")

series-level ROC-AUC: 0.5152  (n=500, break_rate=0.520)


## Eval — TS-AUC (per-series score broadcast across online steps)

In [19]:
pred = pd.DataFrame({"prediction": rows_pred}, index=pd.MultiIndex.from_arrays(
    [rows_id, rows_time], names=["id", "time"])).sort_index()
ytar = pd.read_parquet(f"{PATH}y_train.parquet")           # DataFrame, col "target"
ytar = ytar.loc[ytar.index.get_level_values("id").isin(id_set)]
print(f"TS-AUC (broadcast): {local_ts_auc(ytar, pred):.4f}   vs bank baseline 0.6064")

TS-AUC (broadcast): 0.5290   vs bank baseline 0.6064
